# Assignment 3 

## Apply LDA, QDA and Naive Bayes on Adult data set. Apply k fold and hold out-cross validation techniques.

The target is whether annual income is greater than 50K. The primary metrics are accuracy, sensitivity/recall, specificity, precision, F1-score, and ROC-AUC. Because the classes are imbalanced, accuracy is not interpreted alone.

In [1]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import friedmanchisquare, ttest_rel, wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (StratifiedKFold, cross_validate, train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

#### Loading the Adult dataset

The Adult data are loaded directly from the UCI Machine Learning Repository (`adult.zip`). No local cache, OpenML mirror, or alternate data source is used. The archive's `adult.data` and `adult.test` files are combined, and missing values represented by `?` are retained for pipeline imputation.

In [2]:
from io import BytesIO
from zipfile import ZipFile
from urllib.request import urlopen

adult_columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]
uci_adult_url = "https://archive.ics.uci.edu/static/public/2/adult.zip"
with urlopen(uci_adult_url) as response:
    with ZipFile(BytesIO(response.read())) as archive:
        with archive.open("adult.data") as train_file:
            adult_train = pd.read_csv(train_file, names=adult_columns, skipinitialspace=True, na_values="?")
        with archive.open("adult.test") as test_file:
            adult_test = pd.read_csv(test_file, names=adult_columns, skiprows=1, skipinitialspace=True, na_values="?")
adult = pd.concat([adult_train, adult_test], ignore_index=True)
adult["income"] = adult["income"].astype(str).str.strip().str.replace(".", "", regex=False)
y = (adult.pop("income") == ">50K").astype(int)
X = adult
print("source: UCI Machine Learning Repository — Adult dataset")
print("shape:", X.shape)
print("class counts:", y.value_counts().sort_index().to_dict())
print("missing values:", int(X.isna().sum().sum()))


source: UCI Machine Learning Repository — Adult dataset
shape: (48842, 14)
class counts: {0: 37155, 1: 11687}
missing values: 6465


### (a) Report the preprocessing techniques used, if any.

1. The target is converted to a binary numerical variable: `1` for income above 50K and `0` otherwise.
2. The string `?` is treated as missing data.
3. Numeric columns use median imputation, followed by standardization to mean 0 and standard deviation 1.
4. Categorical columns use most-frequent imputation, followed by one-hot encoding with `handle_unknown="ignore"`.
5. All preprocessing is inside a scikit-learn `Pipeline`, so each training fold learns its imputation, category levels, and scaling parameters independently. This prevents validation/test leakage.

In [3]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("numeric", numeric_pipe, numeric_features), ("categorical", categorical_pipe, categorical_features)], remainder="drop", sparse_threshold=0)
print("numeric features:", numeric_features)
print("categorical features:", categorical_features)

numeric features: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical features: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


#### Models

- **LDA** uses one shared covariance matrix for both classes, producing a linear decision boundary.
- **QDA** estimates one covariance matrix per class, producing a quadratic boundary. A small `reg_param` stabilizes covariance estimates after one-hot encoding.
- **Gaussian Naive Bayes** assumes conditional independence of features given the class and fits a Gaussian distribution per feature and class.

In [4]:
models = {
    "LDA": Pipeline([("preprocess", preprocessor), ("model", LinearDiscriminantAnalysis())]),
    "QDA": Pipeline([("preprocess", preprocessor), ("model", QuadraticDiscriminantAnalysis(reg_param=0.1))]),
    "Naive Bayes": Pipeline([("preprocess", preprocessor), ("model", GaussianNB())]),
}
scoring = {
    "accuracy": "accuracy",
    "sensitivity": "recall",
    "precision": "precision",
    "f1": "f1",
    "roc_auc": "roc_auc",
}
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp) if tn + fp else 0.0


### (b) Report the cross validation technique used.

Two validation techniques are used:

- **Stratified 5-fold cross-validation:** the data are divided into five folds while preserving the income-class ratio. Each fold is used once for validation and the other four folds for training. The reported mean and standard deviation summarize performance across folds.
- **Stratified holdout validation:** 20% of the data are kept as a final test set. The test set is not used during fitting or model comparison; it is evaluated once after training on the 80% training split.

The same preprocessing-and-model pipeline is refitted inside every fold and on the holdout training data.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, estimator in models.items():
    result = cross_validate(estimator, X_train, y_train, cv=skf, scoring=scoring,
                            n_jobs=-1, return_train_score=False)
    for fold in range(5):
        row = {"model": name, "fold": fold + 1}
        row.update({metric: result[f"test_{metric}"][fold] for metric in scoring})
        cv_rows.append(row)
cv_results = pd.DataFrame(cv_rows)
cv_summary = cv_results.groupby("model")[list(scoring)].agg(["mean", "std"]).round(4)
cv_summary

accuracy         sensitivity         precision              f1  \
                mean     std        mean     std      mean     std    mean   
model                                                                        
LDA           0.8415  0.0042      0.5598  0.0152    0.7157  0.0096  0.6282   
Naive Bayes   0.5696  0.0204      0.9389  0.0099    0.3511  0.0101  0.5110   
QDA           0.8365  0.0012      0.7102  0.0055    0.6434  0.0024  0.6752   

                    roc_auc          
                std    mean     std  
model                                
LDA          0.0122  0.8924  0.0020  
Naive Bayes  0.0093  0.7874  0.0263  
QDA          0.0030  0.8957  0.0025

### (c) Report the value of the performance measures used.

For the positive class (`>50K`):

- Sensitivity/recall: $TP/(TP+FN)$
- Specificity: $TN/(TN+FP)$
- Precision: $TP/(TP+FP)$
- F1-score: harmonic mean of precision and sensitivity
- Accuracy: $(TP+TN)/(TP+TN+FP+FN)$
- ROC-AUC: threshold-independent ranking performance

The following table reports holdout metrics, fitting each complete pipeline only on the training partition.

In [6]:
holdout_rows = []
fit_times = {}
for name, estimator in models.items():
    start = time.perf_counter()
    estimator.fit(X_train, y_train)
    fit_times[name] = time.perf_counter() - start
    pred = estimator.predict(X_test)
    probability = estimator.predict_proba(X_test)[:, 1]
    base = {
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "sensitivity": recall_score(y_test, pred),
        "specificity": specificity_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, probability),
        "fit_time_seconds": fit_times[name],
    }
    holdout_rows.append(base)
holdout_results = pd.DataFrame(holdout_rows).set_index("model").round(4)
holdout_results

,accuracy,sensitivity,specificity,precision,f1,roc_auc,fit_time_seconds
model,,,,,,,
LDA,0.8416,0.5569,0.9312,0.7181,0.6273,0.8902,0.4773
QDA,0.8359,0.7169,0.8734,0.6404,0.6765,0.8949,0.5057
Naive Bayes,0.5715,0.9376,0.4563,0.3517,0.5116,0.7949,0.1902


### (d) Compare the performance of the algorithms by using statistical measures

The folds are paired because all algorithms are evaluated on the same validation folds. I compare models using:

1. mean and standard deviation across folds;
2. paired t-tests between each pair using fold-wise F1 scores;
3. Wilcoxon signed-rank tests as a non-parametric paired alternative; and
4. the Friedman test across all three algorithms.

A small p-value indicates evidence that the observed performance differences are unlikely under equal-performance assumptions. Statistical significance should be interpreted together with effect size and practical differences in the mean metrics.

In [7]:
def paired_tests(metric="f1"):
    wide = cv_results.pivot(index="fold", columns="model", values=metric)
    rows = []
    names = list(wide.columns)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = wide[names[i]], wide[names[j]]
            t_stat, t_p = ttest_rel(a, b)
            try:
                w_stat, w_p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
            except ValueError:
                w_stat, w_p = np.nan, np.nan
            rows.append({"comparison": f"{names[i]} vs {names[j]}", "mean_difference": (a-b).mean(), "paired_t_p": t_p, "wilcoxon_p": w_p})
    return pd.DataFrame(rows).round(5)

print("Paired tests on fold-wise F1")
display(paired_tests("f1"))
friedman = friedmanchisquare(*[cv_results.loc[cv_results.model == name, "f1"] for name in models])
print({"friedman_statistic": friedman.statistic, "friedman_p_value": friedman.pvalue})
print("Fold-wise F1 means:")
display(cv_results.groupby("model")["f1"].agg(["mean", "std"]).sort_values("mean", ascending=False).round(4))

Paired tests on fold-wise F1


,comparison,mean_difference,paired_t_p,wilcoxon_p
0,LDA vs Naive Bayes,0.11721,0.00019,0.0625
1,LDA vs QDA,-0.04698,0.00039,0.0625
2,Naive Bayes vs QDA,-0.16419,0.00000,0.0625


{'friedman_statistic': np.float64(10.0), 'friedman_p_value': np.float64(0.006737946999085468)}
Fold-wise F1 means:


,mean,std
model,,
QDA,0.6752,0.0030
LDA,0.6282,0.0122
Naive Bayes,0.5110,0.0093
